# 01: Evaluation metrics, from counting to curves

**Practice set:** `practice/01-metrics.md` · **Exam task:** Jan 18, Section B1 (Code Triage) · **Moves:** M1, M3, M4, M5

## Hook

You train a detector for your arm's gripper camera: *"is the target object in this frame?"* On your test set it scores **95% accuracy**. You put it on the robot, and it keeps closing the gripper on empty air, again and again.

Nothing is broken. The model does exactly what its numbers say. Those numbers just weren't the right ones.

By the end of this notebook you'll be able to say *exactly* why that happens. You'll also be able to read any confusion matrix, threshold, ROC curve or "capacity" question the exam throws at you, by **counting** rather than memorising formulas.

## Before you start

- **You need:** fractions and percentages. Nothing else.
- **How to use this notebook:** run the cells in order. When you see **PREDICT**, write your guess in the cell *before* running the next one. When you see **BUILD**, write the function yourself; the test cell tells you when it's right. Hints sit in collapsed blocks. Open one only after a real attempt.
- **Pen and paper stay on the desk.** The exam has no computer. The notebook is for checking and exploring *after* you've reasoned it out.

In [ ]:
import prep
import numpy as np
import matplotlib.pyplot as plt

---
## 1. Every prediction lands in one of four boxes

**Idea.** A yes/no model makes two kinds of mistakes, not one:
- it can say **yes** when the truth is **no** (a false alarm), or
- it can say **no** when the truth is **yes** (a miss).

Name every outcome by two words: was the model *right* (True/False), and what did it *say* (Positive/Negative)?

| | model says YES | model says NO |
|---|---|---|
| **truth is YES** | TP (true positive) | FN (false negative, a miss) |
| **truth is NO** | FP (false positive, a false alarm) | TN (true negative) |

**Smallest example.** Ten camera frames. `truth` = 1 means the target really is there; `said` is what the model output.

In [ ]:
truth = [1, 0, 0, 1, 0, 0, 0, 1, 0, 0]
said  = [1, 1, 0, 0, 0, 0, 1, 1, 0, 0]

rows = []
for i, (t, s) in enumerate(zip(truth, said), start=1):
    box = {(1, 1): "TP", (1, 0): "FN", (0, 1): "FP", (0, 0): "TN"}[(t, s)]
    rows.append([i, t, s, box])
prep.table(rows, header=["frame", "truth", "said", "box"])

**PREDICT:** Before running the next cell, count each box from the table yourself. TP = ?, FP = ?, FN = ?, TN = ?

In [ ]:
from collections import Counter
print(Counter(r[3] for r in rows))

### BUILD 1: `confusion_counts`
Write a function that returns `(tp, fp, fn, tn)` for two equal-length lists of 0/1 values.

In [ ]:
def confusion_counts(truth, said):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(confusion_counts, [
    ((truth, said), (2, 2, 1, 5)),
    (([1, 1, 1], [1, 1, 1]), (3, 0, 0, 0)),
    (([0, 0, 1, 1], [1, 0, 0, 1]), (1, 1, 1, 1)),
])

---
## 2. Every rate is "a count ÷ a group". The only question is *which group?*

**Idea.** Each metric divides one box by the size of some group. Get the group right and the formula takes care of itself.

| Metric | Question it answers | Count ÷ group |
|---|---|---|
| **Recall** (= sensitivity = TPR) | Of the frames that *truly* have the target, how many did we catch? | TP ÷ (TP + FN), i.e. ÷ all truly-YES |
| **Specificity** (= TNR) | Of the frames that *truly* don't, how many did we correctly leave alone? | TN ÷ (TN + FP), i.e. ÷ all truly-NO |
| **FPR** = 1 − specificity | Of the truly-NO frames, how many did we raise a false alarm on? | FP ÷ (TN + FP) |
| **Precision** | Of the frames we *said* YES on, how many were right? | TP ÷ (TP + FP), i.e. ÷ all said-YES |
| **Accuracy** | Of *all* frames, how many were right? | (TP + TN) ÷ everything |
| **F1** | One number that is high only if precision *and* recall are both high | 2·P·R ÷ (P + R) |

Notice the pattern:
- recall, specificity and FPR divide by a **truth** group (a row of the table above);
- precision divides by a **prediction** group (a column).

Hold on to that. It is the aha of this whole notebook, and you'll see why in section 3.

### BUILD 2: `rates`
Return a dict with keys `recall`, `specificity`, `fpr`, `precision`, `accuracy`, `f1`.

In [ ]:
def rates(tp, fp, fn, tn):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(rates, [
    ((2, 2, 1, 5), {"recall": 2/3, "specificity": 5/7, "fpr": 2/7, "precision": 0.5, "accuracy": 0.7, "f1": 4/7}),
    ((90, 10, 10, 890), {"recall": 0.9, "precision": 0.9, "accuracy": 0.98}),
])

---
## 3. The gripper mystery: prevalence

Your detector has **recall 0.90** and **specificity 0.95**. That sounds great. In the workshop, though, only **2%** of frames actually contain the target.

**PREDICT** (write a number): of the times the gripper closes (the model says YES), roughly what fraction actually have the target?

<details><summary>Hint 1</summary>

Don't use formulas yet. Imagine exactly 1,000 frames. How many truly contain the target?
</details>

<details><summary>Hint 2</summary>

20 frames contain the target and 980 don't. The model catches 90% of the 20. It falsely fires on 5% of the 980. Count both.
</details>

In [ ]:
# Work it out on paper first, then fill this in as a check.
frames = 1000
target_frames = 0.02 * frames
empty_frames = frames - target_frames
tp = 0.90 * target_frames
fp = 0.05 * empty_frames
print(f"truly target: {target_frames:.0f}, truly empty: {empty_frames:.0f}")
print(f"gripper closes on target: {tp:.0f}, on empty air: {fp:.0f}")
print(f"precision = {tp:.0f} / {tp + fp:.0f} = {tp / (tp + fp):.3f}")

Roughly **3 out of 4 grabs are at empty air**, and the model is working exactly as specified. There are so many empty frames that even a 5% false-alarm rate produces more false alarms than there are real targets.

And the "95% accuracy"? Check it: (18 + 931) / 1000 ≈ 0.95. Accuracy looks great *because* almost everything is empty. A model that always says NO would score 98%.

### EXPLORE: what moves when prevalence changes?
**PREDICT:** as prevalence goes from 1% to 50%, which of recall, specificity and precision stay flat? Write your answer, *then* run the cell.

In [ ]:
prev = np.linspace(0.01, 0.5, 50)
recall_ = np.full_like(prev, 0.90)
spec_ = np.full_like(prev, 0.95)
prec_ = (0.90 * prev) / (0.90 * prev + 0.05 * (1 - prev))
plt.figure(figsize=(6, 3.5))
plt.plot(prev, recall_, label="recall (÷ truly-YES)")
plt.plot(prev, spec_, label="specificity (÷ truly-NO)")
plt.plot(prev, prec_, label="precision (÷ said-YES)")
plt.xlabel("prevalence (fraction of frames with the target)")
plt.ylim(0, 1.02); plt.legend(); plt.grid(alpha=.3); plt.title("Same model, different worlds")
plt.show()

**Why?** Recall only looks *inside* the truly-YES group. Making that group bigger or smaller doesn't change what fraction of it gets caught. Precision mixes both groups, so it moves.

**Now try practice problem 1.1 on paper**, then check it:

In [ ]:
prep.check("1.1P", ...)   # floor(100 x precision) in City P
prep.check("1.1Q", ...)   # ... in City Q
prep.check("1.1f", ...)   # follow-up letter

---
## 4. Scores, thresholds and the ROC curve

**Idea.** Real models don't output YES/NO. They output a **score**: higher means "more likely YES". You choose a **threshold**: say YES if score ≥ threshold.
- A low threshold says YES a lot: you catch more targets (TPR ↑) but raise more false alarms (FPR ↑).
- A high threshold is cautious: both go down.

The **ROC curve** plots (FPR, TPR) for *every* possible threshold. It's the whole menu of trade-offs the model offers.

**Smallest example** (the scores from practice problem 1.2):

In [ ]:
pos = [0.90, 0.80, 0.55, 0.30]          # scores of truly-YES items
neg = [0.70, 0.60, 0.40, 0.20, 0.10]    # scores of truly-NO items

rows = []
for thr in sorted(set(pos + neg) | {1.01}, reverse=True):
    tpr = sum(s >= thr for s in pos) / len(pos)
    fpr = sum(s >= thr for s in neg) / len(neg)
    rows.append([thr, tpr, fpr])
prep.table(rows, header=["threshold", "TPR", "FPR"])

**PREDICT:** as the threshold drops one score at a time, each step moves the point either straight **up** or straight **right**. Which kind of score makes it go up, and which makes it go right?

<details><summary>Check your reasoning</summary>

Passing a **positive's** score adds one more caught positive, so TPR goes up by 1/4. Passing a **negative's** score adds one more false alarm, so FPR goes right by 1/5. The ROC curve is a staircase that walks through the scores from highest to lowest.
</details>

### BUILD 3: `roc_points`
Return a list of (fpr, tpr) points. Start with (0, 0) (threshold above every score). Then add one point per **distinct** score, using it as the threshold, from highest to lowest. The last point is always (1, 1).

In [ ]:
def roc_points(pos, neg):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(roc_points, [
    (([0.9, 0.1], [0.5]), [(0.0, 0.0), (0.0, 0.5), (1.0, 0.5), (1.0, 1.0)]),
    (([0.8], [0.2, 0.6]), [(0.0, 0.0), (0.0, 1.0), (0.5, 1.0), (1.0, 1.0)]),
])

In [ ]:
if prep.ready(roc_points, [1], [0]):
    pts = roc_points(pos, neg)
    xs, ys = zip(*pts)
    plt.figure(figsize=(4, 4))
    plt.step(xs, ys, where="post", marker="o")
    plt.plot([0, 1], [0, 1], "--", color="gray", label="coin flip")
    plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC staircase for problem 1.2"); plt.legend(); plt.grid(alpha=.3)
    plt.show()

---
## 5. AUC is just counting pairs

**AUC** = area under the ROC curve. That sounds like calculus. It's really counting.

**PREDICT:** in the staircase above, every little up-step is 1/4 tall and every right-step is 1/5 wide. So the area is made of rectangles of size (1/5 × 1/4) = 1/20. There are 4 × 5 = 20 (positive, negative) pairs. Coincidence?

<details><summary>Hint</summary>

Look only at the **right-steps**. Each one happens when the threshold passes a *negative's* score, and it is 1/5 wide. How tall is the curve at that moment? It equals the fraction of positives *already passed*, which means the positives scoring **above** this negative. So each right-step adds (number of positives that beat this negative) × 1/20 to the area.
</details>

**The aha:** AUC = (number of (pos, neg) pairs where the positive scores higher) ÷ (number of pairs), with a tie counting ½. Equivalently, it's the probability that a random positive outscores a random negative.

### BUILD 4: `auc_by_pairs`

In [ ]:
def auc_by_pairs(pos, neg):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(auc_by_pairs, [
    (([0.9, 0.1], [0.5]), 0.5),
    (([0.8], [0.2, 0.6]), 1.0),
    (([0.5], [0.5]), 0.5),
])

**EXPLORE:** check that the area under your staircase equals the pair count, on random data. (Area of a staircase = sum of width × height of each right-step.)

In [ ]:
if prep.ready(auc_by_pairs, [1], [0]) and prep.ready(roc_points, [1], [0]):
    rng = np.random.default_rng(0)
    for trial in range(5):
        p = list(rng.normal(1, 1, 30).round(3)); n = list(rng.normal(0, 1, 40).round(3))
        pts = roc_points(p, n)
        area = sum((x2 - x1) * y1 for (x1, y1), (x2, y2) in zip(pts, pts[1:]))
        print(f"trial {trial}: staircase area = {area:.4f}   pair count = {auc_by_pairs(p, n):.4f}")

(If ties appear, the staircase has diagonal steps; this simple area formula ignores those, so tiny differences can show up. With continuous random scores ties are rare.)

**PREDICT**, then run: what happens to AUC if every score is squared? Cubed? Replaced by (s − 0.5)²? Write three guesses.

In [ ]:
if prep.ready(auc_by_pairs, [1], [0]):
    for name, f in [("s", lambda s: s), ("s^2", lambda s: s**2), ("s^3", lambda s: s**3),
                    ("(s-0.5)^2", lambda s: round((s - 0.5)**2, 6))]:
        print(f"{name:>10}: AUC = {auc_by_pairs([f(s) for s in pos], [f(s) for s in neg]):.3f}")

**Why:** AUC only asks "who is bigger?" for each pair.
- A **monotone increasing** function (if a > b then f(a) > f(b)) never changes who is bigger, so AUC can't change.
- (s − 0.5)² is *not* monotone. It folds the scale at 0.5, so 0.9 and 0.1 both become 0.16, and the order gets scrambled.

Now do practice problem 1.2 on paper and check:

In [ ]:
prep.check("1.2a", ...)
prep.check("1.2b", ...)
prep.check("1.2c", ...)

---
## 6. Capacity questions: turn rates into counts

The exam likes this pattern: "the team can only handle *K* flagged items per day; which operating point?" A rate means nothing until you multiply it by the size of its group.

**flagged per day = TPR × (number truly YES) + FPR × (number truly NO)**

Try practice problem 1.3 on paper. Then use this cell as a calculator if you like:

In [ ]:
n_yes, n_no = 100, 9900          # per day
points = {"P": (0.005, 0.50), "Q": (0.01, 0.65), "R": (0.02, 0.80), "S": (0.03, 0.88), "T": (0.05, 0.93)}
# write the loop that prints the flagged count for each point (your code here)

In [ ]:
prep.check("1.3", ...)    # letter of the point
prep.check("1.3p", ...)   # floor(100 x precision) at that point

---
## 7. Errors for numbers: MAE vs RMSE

For predictions that are numbers (prices, distances), the error is (prediction − truth).
- **MAE** = average of |error|. Every unit of error counts the same.
- **RMSE** = √(average of error²). Squaring makes big errors count *much* more: an error of 10 counts as 100.

**PREDICT:** Model A misses by 2 every day. Model B is perfect for 4 days, then misses by 10. Which does MAE prefer? Which does RMSE prefer?

Then do practice problem 1.4. For part (b), use the cell below to *see* the answer before you prove it on paper.

In [ ]:
errors_B = np.array([0, 0, 0, 0, 10])
cs = np.linspace(-2, 6, 401)
rmse = [np.sqrt(np.mean((errors_B - c) ** 2)) for c in cs]
mae = [np.mean(np.abs(errors_B - c)) for c in cs]
plt.figure(figsize=(6, 3.5))
plt.plot(cs, rmse, label="RMSE after shifting by c")
plt.plot(cs, mae, label="MAE after shifting by c")
plt.xlabel("c"); plt.legend(); plt.grid(alpha=.3); plt.title("Where is each curve lowest?")
plt.show()

Read off where each curve bottoms out. Compare those two numbers with the **mean** and the **median** of [0, 0, 0, 0, 10]. Then prove it on paper (the hint ladder for 1.4 walks through why).

In [ ]:
prep.check("1.4b_c_rmse", ...)
prep.check("1.4b_c_mae", ...)
prep.check("1.4b_rmse", ...)

---
## 8. Many classes, and the convention trap

With 3 classes, the confusion matrix is 3×3. To get per-class metrics, treat one class as "YES" and the other two as "NO" (*one-vs-rest*), then use exactly the same counting as before.

**The trap:** which axis is *truth*? Many textbooks put truth on the rows. The Jan 18 paper put **predictions on the rows**. Always read the definition given in the question, never the one you remember (move **M1**).

### BUILD 5: `per_class`
`M` is a list of lists. `rows_are` is `"actual"` or `"predicted"`. Return a list of `(recall, precision)` per class.

In [ ]:
def per_class(M, rows_are):
    # your code here
    raise NotImplementedError

In [ ]:
prep.test(per_class, [
    (([[8, 2], [1, 9]], "actual"), [(0.8, 8/9), (0.9, 9/11)]),
    (([[8, 2], [1, 9]], "predicted"), [(8/9, 0.8), (9/11, 0.9)]),
])

Now do practice problem 1.5 on paper (watch the convention!) and check:

In [ ]:
prep.check("1.5a_recall", ...)
prep.check("1.5a_precision", ...)
prep.check("1.5b", ...)
prep.check("1.5c", ...)

---
## 9. Exam walkthrough: Jan 18, Section B1 (Code Triage)

> Do this **after** taking the Jan 18 paper as your diagnostic. Open `papers/2026-stage1-jan18/paper.pdf`, pages 10–14.

B1 gives you class sizes (60% / 30% / 10% of 1,000 patients), each class's sensitivity and specificity, and two facts about "egregious errors". **Every** later question uses the full 3×3 table of counts. So the real task is to **rebuild that table** (move **M3: fill the whole table**).

### Step 1: what are you building?
Draw a blank 3×3 grid on paper with rows = **actual** class 1, 2, 3 and columns = **predicted** class 1, 2, 3. How many unknown cells are there, and how many facts do you have?

<details><summary>Hint</summary>

There are 9 unknowns. You know the 3 row totals (600, 300, 100), 3 sensitivities, 3 specificities, and 2 egregious-error facts. That's more facts than unknowns, so some facts will be *checks*.
</details>

### Step 2: the diagonal
Sensitivity of class *i* = (correct class-*i* predictions) ÷ (actual class *i*). Fill in the three diagonal cells.

### Step 3: the egregious errors
"3.33% of actual class 1 → predicted 3" and "10% of actual class 3 → predicted 1". Fill in those two cells. Each of rows 1 and 3 now has exactly one unknown left, and the row totals give it to you.

<details><summary>Hint</summary>

Row 1: 600 = (diagonal) + (1→2) + (1→3). You know two of the three.
</details>

### Step 4: the last row
Row 2 has two unknowns: (2→1) and (2→3). Which specificity mentions predictions *into* class 3 from people who are *not* class 3?

<details><summary>Hint</summary>

Specificity₃ = (non-class-3 people **not** predicted 3) ÷ 900. So the number of non-class-3 people predicted 3 is 900 × (1 − 0.91). Part of that comes from row 1, which you already know.
</details>

### Step 5: fill in your table and let the code check it against the facts you did *not* use

In [ ]:
# rows = actual class 1,2,3 ; columns = predicted class 1,2,3
my_table = [
    [..., ..., ...],
    [..., ..., ...],
    [..., ..., ...],
]

def check_triage(T):
    if any(x is ... for row in T for x in row):
        print("Fill in all nine cells first."); return
    T = np.array(T, dtype=float)
    n = T.sum(axis=1)
    print("row totals (should be 600, 300, 100):", n)
    for i in range(3):
        sens = T[i, i] / n[i]
        others = [j for j in range(3) if j != i]
        spec = sum(T[j, k] for j in others for k in others if k != i) / n[others].sum()
        print(f"class {i+1}: sensitivity {sens:.3f}, specificity {spec:.3f}")
    print("given: sens 0.85/0.70/0.80, spec 0.90/0.89/0.91 (specificities rounded to 2 decimals)")

check_triage(my_table)

If all three specificities match after rounding, your table is right, including the two you never used. That's what makes M3 powerful: the extra facts check your work for free.

**One subtlety:** the paper prints specificities rounded to 2 decimals. That leaves a little slack. Two slightly different splits of row 2 both round to the printed values, and every question's answer comes out the same either way. If your table passes all the checks but differs by one or two from a friend's, rounding is why.

### Step 6: answer the questions from your table
- **Q1** (missed ICU patients) and **Q2** (outpatients correctly kept out): read them straight off your table.
- **Q3** (25 ICU beds): which section of this notebook is this? Turn rates into counts for 200 patients a day.
- **Q4** (score → score^1.5) and **Q5** (more ICU patients): which section answered these? You already know why.
- **Q7** (benefit matrix): multiply each cell of your table by its benefit, add everything, divide by 1,000.

In [ ]:
prep.check("J18-B1-Q1", ...)
prep.check("J18-B1-Q2", ...)
prep.check("J18-B1-Q3", ...)   # option number
prep.check("J18-B1-Q4", ...)   # option number
prep.check("J18-B1-Q5", ...)   # option number
prep.check("J18-B1-Q7", ...)

Q6 (the claims about architectures and losses) is about reasoning, not counting. Work through it with the hint ladder in `papers/2026-stage1-jan18/SOLUTIONS.md`, then check:

In [ ]:
prep.check("J18-B1-Q6", ...)

---
## 10. Connect

Answer in your own words in the cell below. Then copy the best sentence into `log/ERROR_LOG.md` under "Moves I own".

1. In problem 1.1 and in B1-Q5, the ROC curve didn't change when the mix of classes changed. In 1.2 it didn't change when scores were squared. What single fact explains *both*?
2. AUC only cares about the **order** of scores. In Set 04 you'll meet a metric that also only cares about order (Precision@k). Guess now: would squaring all scores change Precision@k?
3. In 1.4, squared error was minimised by the **mean** and absolute error by the **median**. Look at the Jan 18 "Godown" problem: which distance gives which?

*Your answers here.*